In [11]:
import os

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import torch
import numpy as np
import random

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
import yaml

BASE_PATH = "/content/drive/MyDrive/mosquito-robustness"

with open(os.path.join(BASE_PATH, "configs/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
noise_levels = config["noise"]["sigmas"]
batch_size = config["training"]["batch_size"]

In [15]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"

image_size = config["data"]["image_size"]

test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        config["normalization"]["mean"],
        config["normalization"]["std"]
    )
])

base_dir = "/content/images"
test_dir = os.path.join(base_dir, "testing")

test_dataset = datasets.ImageFolder(test_dir, transform=test_transforms)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

replace /content/images/testing/Aedes Aegypti/Aedes Aegypti_401.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [16]:
# eff_model, res_model, swin_model already defined (NOTEBOOK 3 CODES)



BASE_PATH = "/content/drive/MyDrive/mosquito-robustness"

with open(os.path.join(BASE_PATH, "configs/config.yaml"), "r") as f:
    config = yaml.safe_load(f)



import torch.nn as nn
from torchvision import models

!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"
train_dir = "/content/images/training"
num_classes = len(os.listdir(train_dir))

print("Number of classes:", num_classes)

#efficientnet
def get_efficientnet(num_classes):
    model = models.efficientnet_b0(weights="IMAGENET1K_V1")

    in_features = model.classifier[1].in_features

    # Replace classifier
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for param in model.features.parameters():
        param.requires_grad = False

    return model


def get_resnet(num_classes):
    model = models.resnet50(weights="IMAGENET1K_V1")

    in_features = model.fc.in_features

    model.fc = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad = False

    return model


def get_swin(num_classes):
    model = models.swin_v2_t(weights="IMAGENET1K_V1")

    in_features = model.head.in_features

    model.head = nn.Linear(in_features, num_classes)

    # Freeze backbone
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False

    return model


class ModelWithEmbedding(nn.Module):
    def __init__(self, model, model_name):
        super().__init__()
        self.model = model
        self.model_name = model_name

    def forward(self, x):
        if self.model_name == "resnet":
            # Extract features before FC
            features = self.model.avgpool(self.model.layer4(
                        self.model.layer3(
                        self.model.layer2(
                        self.model.layer1(
                        self.model.maxpool(
                        self.model.relu(
                        self.model.bn1(
                        self.model.conv1(x)))))))))
            features = torch.flatten(features, 1)
            logits = self.model.fc(features)

        elif self.model_name == "efficientnet":
            features = self.model.features(x)
            features = self.model.avgpool(features)
            features = torch.flatten(features, 1)
            logits = self.model.classifier(features)

        elif self.model_name == "swin":

            x=self.model.features(x)

            x=self.model.norm(x)

            features = x.mean(dim=(1,2))

            logits = self.model.head(features)
        return features, logits


eff_model, eff_dim = get_efficientnet(num_classes)
res_model, res_dim = get_resnet(num_classes)
swin_model, swin_dim = get_swin(num_classes)

eff_model = ModelWithEmbedding(eff_model, "efficientnet").to(DEVICE)
res_model = ModelWithEmbedding(res_model, "resnet").to(DEVICE)
swin_model = ModelWithEmbedding(swin_model, "swin").to(DEVICE)

print("Models initialized.")




Using device: cuda
replace /content/images/testing/Aedes Aegypti/Aedes Aegypti_401.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Number of classes: 6
Models initialized.


In [17]:
eff_model.load_state_dict(torch.load("/content/drive/MyDrive/mosquito-robustness/models/efficientnet.pth"))
res_model.load_state_dict(torch.load("/content/drive/MyDrive/mosquito-robustness/models/resnet.pth"))
swin_model.load_state_dict(torch.load("/content/drive/MyDrive/mosquito-robustness/models/swin.pth"))

eff_model.to(device).eval()
res_model.to(device).eval()
swin_model.to(device).eval()

ModelWithEmbedding(
  (model): SwinTransformer(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (1): Permute()
        (2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      )
      (1): Sequential(
        (0): SwinTransformerBlockV2(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): ShiftedWindowAttentionV2(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (cpb_mlp): Sequential(
              (0): Linear(in_features=2, out_features=512, bias=True)
              (1): ReLU(inplace=True)
              (2): Linear(in_features=512, out_features=3, bias=False)
            )
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): MLP(
            (0): Linear(in_

In [18]:
def add_gaussian_noise(images, sigma):
    noise = torch.randn_like(images) * sigma
    noisy = images + noise
    return torch.clamp(noisy, 0, 1)

In [19]:
def evaluate(model, loader, sigma):
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if sigma > 0:
                images = add_gaussian_noise(images, sigma)

            _, outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [20]:
import json

os.makedirs("results", exist_ok=True)

all_results = {}

models = {
    "efficientnet": eff_model,
    "resnet": res_model,
    "swin": swin_model
}

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    results = {}

    for sigma in noise_levels:
        acc = evaluate(model, test_loader, sigma)
        results[sigma] = acc
        print(f"Sigma {sigma}: {acc:.4f}")

    all_results[name] = results

    with open(f"results/{name}_noise.json", "w") as f:
        json.dump(results, f)


Evaluating efficientnet...
Sigma 0.0: 0.8733
Sigma 0.05: 0.1967
Sigma 0.1: 0.1150
Sigma 0.15: 0.1167
Sigma 0.2: 0.1700

Evaluating resnet...
Sigma 0.0: 0.9000
Sigma 0.05: 0.4683
Sigma 0.1: 0.3933
Sigma 0.15: 0.3633
Sigma 0.2: 0.3783

Evaluating swin...
Sigma 0.0: 0.8933
Sigma 0.05: 0.6167
Sigma 0.1: 0.5933
Sigma 0.15: 0.5733
Sigma 0.2: 0.5350
